1. Leer las predicciones de V1 y V2

In [1]:
library(data.table)
base <- "/content/buckets/b1/exp"
a <- fread(file.path(base, "WF9130", "prediccion.txt"),
           colClasses = c(numero_de_cliente = "character"))
b <- fread(file.path(base, "WF9160", "prediccion.txt"),
           colClasses = c(numero_de_cliente = "character"))


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%




2. Verificar clientes y puntajes

In [2]:
for (x in list(a, b)) {
  stopifnot(all(c("numero_de_cliente", "prob") %in% names(x)),
            is.numeric(x$prob), nrow(x) >= 2400L, !anyNA(x$numero_de_cliente),
            !anyDuplicated(x$numero_de_cliente), all(is.finite(x$prob)))
}
stopifnot(setequal(a$numero_de_cliente, b$numero_de_cliente))

3. Combinar los rankings con peso 50/50

In [3]:
a[, rango30 := frank(-prob, ties.method = "average")]
b[, rango60 := frank(-prob, ties.method = "average")]
tb_prediccion <- merge(a[, .(numero_de_cliente, rango30)],
                       b[, .(numero_de_cliente, rango60)], by = "numero_de_cliente")
# Mayor prob representa mayor prioridad para contactar.
tb_prediccion[, prob := 1 - ((rango30 + rango60) / 2 - 1) / .N]

4. Crear la carpeta nueva y guardar el ranking

In [4]:
stopifnot(nrow(tb_prediccion) >= 2400L)
PARAM <- list(experimento = 9190L)
destino <- file.path(base, "WF9190")
if (!dir.create(destino, showWarnings = FALSE)) stop("Carpeta WF9190 no disponible.")
fwrite(tb_prediccion, file.path(destino, "prediccion.txt"), sep = "\t")
# Generar CSV para revisión y envío manual
cortes <- seq(1800L, 2400L, by = 100L)
stopifnot(nrow(tb_prediccion) >= max(cortes),
          !anyNA(tb_prediccion$numero_de_cliente),
          !anyDuplicated(tb_prediccion$numero_de_cliente),
          all(is.finite(tb_prediccion$prob)))
setorder(tb_prediccion, -prob, numero_de_cliente)
carpeta_kaggle <- file.path(destino, "kaggle")
if (!dir.create(carpeta_kaggle)) stop("No se pudo crear la carpeta kaggle.")

5. Generar los siete CSV para Kaggle

In [5]:
archivos <- file.path(carpeta_kaggle, paste0("KA", PARAM$experimento, "_", cortes, ".csv"))
if (!dir.exists(carpeta_kaggle)) stop("Ejecutá primero la celda 4.")
if (any(file.exists(archivos))) stop("Ya existen CSV para esta corrida.")

for (corte in cortes) {
  salida <- tb_prediccion[, .(numero_de_cliente)]
  salida[, Predicted := as.integer(seq_len(.N) <= corte)]
  fwrite(salida, file.path(carpeta_kaggle, paste0("KA", PARAM$experimento, "_", corte, ".csv")))
}